In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import roc_auc_score
from collections import Counter
from rdkit.Chem import AllChem
from rdkit import Chem
from sklearn import metrics
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import os

In [4]:
def smiles_to_morgan_fp(smiles: str, n_bits: int = 1024, radius: int = 2) -> np.ndarray:
    """
    Converts a SMILES string to a Morgan fingerprint.
    """
    mol = Chem.MolFromSmiles(smiles)    
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

In [16]:
df = pd.read_csv("/home/acomajuncosa/Downloads/activities_CHEMBL4513220.tsv", sep='\t', low_memory=False)
df = df[~df['Smiles'].isna()]
print("All are %: " + str(set([i for i in df['Standard Units']])))


# Get actives and inactives
actives = df[df['Standard Value'] >= 50]['Smiles'].tolist()
inactives = df[df['Standard Value'] < 50]['Smiles'].tolist()
print("Actives: " + str(len(actives)))
print("Inactives: " + str(len(inactives)))

# Fix random seed
np.random.seed(42)

# Choose N actives and 5 * N inactives
N = 2000
selected_actives = np.random.choice(actives, N, replace=False).tolist()
selected_inactives = np.random.choice(inactives, 5 * N, replace=False).tolist()
print("Actives: " + str(len(selected_actives)))
print("Inactives: " + str(len(selected_inactives)))

# Get ECFPs
print("Calculating ECFPs...")
selected_actives = [smiles_to_morgan_fp(i) for i in selected_actives]
selected_inactives = [smiles_to_morgan_fp(i) for i in selected_inactives]

# Create matrices
X = np.array(selected_actives + selected_inactives)
Y = np.array([1]*len(selected_actives) + [0]*len(selected_inactives))
print("Matrix shapes:")
print(X.shape, Y.shape)

All are %: {'%'}
Actives: 2144
Inactives: 62623
Actives: 2000
Inactives: 10000
Calculating ECFPs...
Matrix shapes:
(12000, 1024) (12000,)


In [17]:
# Split into training and test sets (80% train, 20% test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [18]:
# Train a Random Forest Classifier
clf = RandomForestClassifier(n_estimators=10, random_state=24, n_jobs=8)
clf.fit(X_train, Y_train)
random.seed(42)
Y_train_shuffled = np.copy(Y_train)
random.shuffle(Y_train_shuffled)
print(f"AUROC TRAIN: {metrics.roc_auc_score(Y_train, clf.predict_proba(X_train)[:,1])}")
print(f"AUROC TRAIN SHUFFLED: {metrics.roc_auc_score(Y_train_shuffled, clf.predict_proba(X_train)[:,1])}")
print(f"AUROC TEST: {metrics.roc_auc_score(Y_test, clf.predict_proba(X_test)[:,1])}")

AUROC TRAIN: 0.9997008984375001
AUROC TRAIN SHUFFLED: 0.496109453125
AUROC TEST: 0.8249
